In [1]:

import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

file_path = (
    project_root
    / "data/raw/personal_finance/personal_transactions.xlsx"
)

df = pd.read_excel(file_path)

clean_df = df.copy()

print("Original rows:", len(clean_df))
display(clean_df.head())


Original rows: 806


,Date,Description,Amount,Transaction Type,Category,Account Name,Month
0,2018-01-01,Amazon,11.11,debit,Shopping,Platinum Card,2018-01
1,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,Checking,2018-01
2,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,Silver Card,2018-01
3,2018-01-03,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card,2018-01
4,2018-01-04,Netflix,11.76,debit,Movies & Dvds,Platinum Card,2018-01


In [2]:

clean_df = clean_df.rename(columns={
    "Date": "transaction_date",
    "Description": "description",
    "Amount": "amount",
    "Transaction Type": "transaction_type",
    "Category": "category",
    "Account Name": "account_name",
    "Month": "source_month"
})

clean_df["transaction_date"] = pd.to_datetime(
    clean_df["transaction_date"],
    errors="coerce"
)

for col in ["description", "category", "account_name"]:
    clean_df[col] = clean_df[col].astype("string").str.strip()

clean_df["transaction_type"] = (
    clean_df["transaction_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

clean_df["amount"] = pd.to_numeric(
    clean_df["amount"],
    errors="coerce"
)

display(clean_df.head())


,transaction_date,description,amount,transaction_type,category,account_name,source_month
0,2018-01-01,Amazon,11.11,debit,Shopping,Platinum Card,2018-01
1,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,Checking,2018-01
2,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,Silver Card,2018-01
3,2018-01-03,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card,2018-01
4,2018-01-04,Netflix,11.76,debit,Movies & Dvds,Platinum Card,2018-01


In [3]:

clean_df["is_transfer_candidate"] = (
    clean_df["description"]
    .str.casefold()
    .eq("credit card payment")
    .fillna(False)
)

print("Transfer candidates:",
      clean_df["is_transfer_candidate"].sum())

display(
    clean_df.loc[
        clean_df["is_transfer_candidate"],
        ["transaction_date", "account_name",
         "transaction_type", "amount"]
    ].head(10)
)


Transfer candidates: 143


,transaction_date,account_name,transaction_type,amount
3,2018-01-03,Platinum Card,credit,2298.09
22,2018-01-22,Platinum Card,credit,554.99
23,2018-01-22,Silver Card,credit,309.81
24,2018-01-22,Checking,debit,554.99
26,2018-01-23,Checking,debit,309.81
38,2018-02-05,Platinum Card,credit,145.14
39,2018-02-06,Silver Card,credit,154.13
40,2018-02-07,Checking,debit,154.13
58,2018-02-26,Platinum Card,credit,765.37
59,2018-02-26,Silver Card,credit,156.11


In [4]:

subscription_merchants = [
    "netflix",
    "spotify"
]

clean_df["is_subscription_candidate"] = (
    clean_df["description"]
    .str.casefold()
    .isin(subscription_merchants)
    .fillna(False)
)

print(
    "Subscription candidates:",
    clean_df["is_subscription_candidate"].sum()
)

display(
    clean_df.loc[
        clean_df["is_subscription_candidate"],
        ["transaction_date", "description",
         "amount", "category"]
    ].head(15)
)


Subscription candidates: 41


,transaction_date,description,amount,category
4,2018-01-04,Netflix,11.76,Movies & Dvds
9,2018-01-09,Spotify,10.69,Music
37,2018-02-04,Netflix,11.76,Movies & Dvds
43,2018-02-09,Spotify,10.69,Music
71,2018-03-04,Netflix,11.76,Movies & Dvds
81,2018-03-09,Spotify,10.69,Music
119,2018-04-04,Netflix,11.76,Movies & Dvds
124,2018-04-09,Spotify,10.69,Music
160,2018-05-04,Netflix,11.76,Movies & Dvds
167,2018-05-09,Spotify,10.69,Music


In [5]:

import numpy as np

clean_df["signed_amount"] = np.select(
    [
        clean_df["transaction_type"].eq("credit"),
        clean_df["transaction_type"].eq("debit")
    ],
    [
        clean_df["amount"],
        -clean_df["amount"]
    ],
    default=np.nan
)

clean_df["year_month"] = (
    clean_df["transaction_date"]
    .dt.strftime("%Y-%m")
)

display(
    clean_df[
        ["transaction_date", "description",
         "transaction_type", "amount",
         "signed_amount"]
    ].head()
)


,transaction_date,description,transaction_type,amount,signed_amount
0,2018-01-01,Amazon,debit,11.11,-11.11
1,2018-01-02,Mortgage Payment,debit,1247.44,-1247.44
2,2018-01-02,Thai Restaurant,debit,24.22,-24.22
3,2018-01-03,Credit Card Payment,credit,2298.09,2298.09
4,2018-01-04,Netflix,debit,11.76,-11.76


In [6]:

print("Total rows:", len(clean_df))
print("Missing values:")
print(clean_df.isnull().sum())

print("\nInvalid dates:",
      clean_df["transaction_date"].isna().sum())

print("Invalid amounts:",
      clean_df["amount"].isna().sum())

print("Unknown transaction types:",
      (~clean_df["transaction_type"].isin(
          ["credit", "debit"]
      )).sum())

assert len(clean_df) == 806

print("\nInitial validation completed!")


Total rows: 806
Missing values:
transaction_date             0
description                  0
amount                       0
transaction_type             0
category                     0
account_name                 0
source_month                 0
is_transfer_candidate        0
is_subscription_candidate    0
signed_amount                0
year_month                   0
dtype: int64

Invalid dates: 0
Invalid amounts: 0
Unknown transaction types: 0

Initial validation completed!


In [7]:

output_dir = project_root / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    output_dir / "personal_transactions_cleaned.csv"
)

clean_df.to_csv(output_path, index=False)

print("Saved:", output_path)


Saved: /Users/yagna/FinPilotAI/FinPilotAI/data/processed/personal_transactions_cleaned.csv
